# Coqui TTS 한국어 훈련 (Google Colab T4)

**런타임 설정**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

| 항목 | 내용 |
|------|------|
| 프레임워크 | Coqui TTS (VITS) |
| Python | 3.12 호환 (`pip install TTS` 한 줄) |
| GPU | T4 16GB (무료) |
| 체크포인트 | Google Drive 자동 저장 |

## Step 1: GPU 확인

In [ ]:
import torch, subprocess
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️ GPU 없음 → 런타임 유형 변경에서 T4 GPU 선택')

## Step 2: Google Drive 연결

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/coqui-korean'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✓ 출력 경로: {OUTPUT_DIR}')

## Step 2.5: Kaggle 인증 및 KSS 데이터셋 다운로드

### Kaggle 토큰 등록 방법 (최초 1회)
1. Colab 왼쪽 사이드바 **🔑 (Secrets)** 아이콘 클릭
2. **Add new secret** 두 번 추가:
   - Name: `KAGGLE_USERNAME` → Value: 캐글 아이디
   - Name: `KAGGLE_KEY` → Value: 캐글 API 키 (`KGAT_...` 부분)
3. 각각 **Notebook access 토글 ON**
4. 아래 셀 실행

In [ ]:
import os, json, subprocess, sys, getpass, glob

# ── Kaggle 인증 ───────────────────────────────────────────────
try:
    from google.colab import userdata
    kaggle_user = userdata.get('KAGGLE_USERNAME')
    kaggle_key  = userdata.get('KAGGLE_KEY')
    print(f"✓ Secrets에서 인증 ({kaggle_user})")
except Exception:
    print("Secrets 없음 → 직접 입력")
    kaggle_user = input("Kaggle 아이디: ")
    kaggle_key  = getpass.getpass("Kaggle API 키 (KGAT_...): ")

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({"username": kaggle_user, "key": kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print("✓ Kaggle 인증 완료")

# ── KSS 다운로드 ──────────────────────────────────────────────
def find_transcript(root):
    hits = glob.glob(f'{root}/**/transcript*.txt', recursive=True)
    return hits[0] if hits else None

TRANSCRIPT_FILE = find_transcript('/content')
if TRANSCRIPT_FILE:
    print(f"✓ KSS 이미 존재: {TRANSCRIPT_FILE}")
else:
    print("KSS 다운로드 중 (~3GB, 5~10분)...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])
    r = subprocess.run(
        'kaggle datasets download -d bryanpark/korean-single-speaker-speech-dataset '
        '-p /content --unzip',
        shell=True, capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr[-1000:])
        raise RuntimeError("KSS 다운로드 실패")

    TRANSCRIPT_FILE = find_transcript('/content')
    if not TRANSCRIPT_FILE:
        r2 = subprocess.run(
            'find /content -maxdepth 4 \\( -name "*.txt" -o -name "*.csv" \\) | head -20',
            shell=True, capture_output=True, text=True)
        print("발견된 파일:", r2.stdout)
        raise RuntimeError("transcript 파일을 찾을 수 없습니다")
    print(f"✓ 다운로드 완료: {TRANSCRIPT_FILE}")

# ── KSS → LJSpeech 변환 ──────────────────────────────────────
KSS_DIR = os.path.dirname(TRANSCRIPT_FILE)
DATASET_PATH = '/content/recordings'
os.makedirs(f'{DATASET_PATH}/wavs', exist_ok=True)

lines_out, missing = [], 0
with open(TRANSCRIPT_FILE, encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('|')
        if len(parts) < 2:
            continue
        rel_path = parts[0].strip()
        text     = parts[1].strip()
        if not text:
            continue
        candidates = [
            os.path.join(KSS_DIR, rel_path),
            os.path.join('/content', rel_path),
            os.path.join(KSS_DIR, *rel_path.split('/')[-2:]),
        ]
        src = next((p for p in candidates if os.path.exists(p)), None)
        if src:
            fname = rel_path.replace('/', '_').replace('.wav', '')
            dst   = f'{DATASET_PATH}/wavs/{fname}.wav'
            if not os.path.exists(dst):
                os.symlink(src, dst)
            lines_out.append(f'{fname}|{text}|{text}')
        else:
            missing += 1

if not lines_out:
    raise RuntimeError(f"WAV 파일을 못 찾았습니다. KSS_DIR={KSS_DIR}")

with open(f'{DATASET_PATH}/metadata.csv', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines_out))

print(f"✓ KSS 변환 완료: {len(lines_out)}개 문장 (누락: {missing}개)")
print(f"✓ 데이터 경로: {DATASET_PATH}")

## Step 3: 패키지 설치 (Python 3.12 완벽 호환)

In [ ]:
import subprocess, sys

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip'] + list(args),
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(f'pip 실패: {args}')

print(f'Python: {sys.version.split()[0]}')

# espeak-ng (한국어 음소 변환)
subprocess.run('apt-get install -y -qq espeak-ng libespeak-ng-dev', shell=True)
print('✓ espeak-ng')

# Coqui TTS — pip install 한 줄로 끝
pip('install', '-q', 'coqui-tts')
print('✓ Coqui TTS')

# 음소 변환 백엔드
pip('install', '-q', 'phonemizer', 'gruut')
print('✓ phonemizer')

print('\n✓ 모든 패키지 설치 완료')

## Step 4: 설치 검증

In [ ]:
import torch
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import Vits
from trainer import Trainer

print(f'✓ TTS import OK')
print(f'✓ CUDA: {torch.cuda.is_available()}')

# espeak-ng 한국어 테스트
import subprocess
r = subprocess.run(['espeak-ng', '-v', 'ko', '-q', '--ipa', '안녕하세요'],
                   capture_output=True, text=True)
print(f'✓ 한국어 음소: {r.stdout.strip()[:40]}')
print('\n✓ 환경 정상 — Step 5로 진행하세요')

## Step 5: 데이터 확인\n\nStep 2.5에서 KSS를 받았다면 이 셀은 건너뛰세요.  \n샘플 데이터(파이프라인 테스트용)만 쓸 경우 실행합니다.

In [ ]:
import os, math, struct

USE_SAMPLE_DATA = True  # 파이프라인 테스트: True | 내 데이터 사용: False

if USE_SAMPLE_DATA:
    DATASET_PATH = '/content/recordings'
    os.makedirs(f'{DATASET_PATH}/wavs', exist_ok=True)

    TEXTS = [
        '안녕하세요 반갑습니다',
        '오늘 날씨가 정말 좋네요',
        '한국어 음성 합성 테스트입니다',
        '코퀴 TTS로 훈련을 시작합니다',
        '인공지능 기술이 빠르게 발전하고 있습니다',
        '목소리 데이터를 수집하면 더 자연스러워집니다',
        '딥러닝 모델을 학습시키는 중입니다',
        '좋은 결과가 나오기를 기대합니다',
        '자연스러운 한국어 음성을 만들겠습니다',
        '셰르파 온넥스 서버에 연결할 예정입니다',
        '음성 합성 기술은 매우 흥미롭습니다',
        '데이터를 많이 모을수록 품질이 좋아집니다',
        '훈련이 완료되면 바로 테스트해 보겠습니다',
        '클라우드 환경에서 GPU를 사용합니다',
        '구글 코랩에서 무료로 훈련할 수 있습니다',
        '체크포인트는 드라이브에 자동으로 저장됩니다',
        '한국어는 아름다운 언어입니다',
        '오늘도 열심히 공부하겠습니다',
        '기계학습은 미래 기술의 핵심입니다',
        '좋은 하루 보내세요',
    ]

    def make_wav(path, freq=220, sr=22050, dur=2.0):
        n = int(sr * dur)
        samples = [int(32767 * math.sin(2 * math.pi * freq * i / sr)) for i in range(n)]
        with open(path, 'wb') as f:
            f.write(b'RIFF'); f.write(struct.pack('<I', 36 + n*2))
            f.write(b'WAVEfmt '); f.write(struct.pack('<IHHIIHH', 16,1,1,sr,sr*2,2,16))
            f.write(b'data'); f.write(struct.pack('<I', n*2))
            f.write(struct.pack(f'<{n}h', *samples))

    lines = []
    for i, text in enumerate(TEXTS):
        fname = f'{i+1:04d}'
        make_wav(f'{DATASET_PATH}/wavs/{fname}.wav', freq=200+i*10)
        lines.append(f'{fname}|{text}|{text}')
    with open(f'{DATASET_PATH}/metadata.csv', 'w') as f:
        f.write('\n'.join(lines))

    print(f'✓ 샘플 데이터 {len(TEXTS)}개 생성 ({DATASET_PATH})')
    print('⚠️  파이프라인 테스트용 — 실제 훈련엔 진짜 음성 데이터 필요')

else:
    DATASET_PATH = f'{OUTPUT_DIR}/recordings'
    assert os.path.exists(f'{DATASET_PATH}/metadata.csv'), \
        f'metadata.csv 없음: {DATASET_PATH}'
    with open(f'{DATASET_PATH}/metadata.csv') as f:
        first = f.readline().strip()
    if len(first.split('|')) < 3:
        print('⚠️  metadata.csv 2컬럼 → 3컬럼 변환 중...')
        lines = open(f'{DATASET_PATH}/metadata.csv').readlines()
        with open(f'{DATASET_PATH}/metadata.csv', 'w') as f:
            for line in lines:
                p = line.strip().split('|')
                f.write(f'{p[0]}|{p[1]}|{p[1]}\n' if len(p) == 2 else line)
        print('✓ 변환 완료')
    n = sum(1 for _ in open(f'{DATASET_PATH}/metadata.csv'))
    print(f'✓ 데이터: {n}개 문장')

## Step 6: 훈련 설정 및 실행

In [ ]:
import os, torch, glob
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import Vits, VitsAudioConfig
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor

dataset_config = BaseDatasetConfig(
    formatter='ljspeech',
    meta_file_train='metadata.csv',
    path=DATASET_PATH,
    language='ko'
)

audio_config = VitsAudioConfig(
    sample_rate=22050, win_length=1024, hop_length=256,
    num_mels=80, mel_fmin=0, mel_fmax=None
)

config = VitsConfig(
    audio=audio_config,
    run_name='vits-korean',
    batch_size=16,
    eval_batch_size=8,
    batch_group_size=5,
    num_loader_workers=2,
    num_eval_loader_workers=2,
    run_eval=True,
    test_delay_epochs=-1,
    epochs=5000,
    text_cleaner='phoneme_cleaners',
    use_phonemes=True,
    phoneme_language='ko',
    phoneme_cache_path=os.path.join(OUTPUT_DIR, 'phoneme_cache'),
    compute_input_seq_cache=True,
    print_step=50,
    print_eval=False,
    mixed_precision=True,
    eval_split_size=0.1,        # 샘플 수 적을 때 검증셋 비율 명시
    eval_split_max_size=256,
    test_sentences=[
        ['안녕하세요 반갑습니다.'],
        ['오늘 날씨가 정말 좋네요.'],
    ],
    output_path=OUTPUT_DIR,
    datasets=[dataset_config],
    save_step=1000,
    save_n_checkpoints=3,
    save_best_after=1000,
)

ap = AudioProcessor.init_from_config(config)
tokenizer, config = TTSTokenizer.init_from_config(config)

train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)
print(f'훈련 샘플: {len(train_samples)}, 검증 샘플: {len(eval_samples)}')

model = Vits(config, ap, tokenizer, speaker_manager=None)

ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/**/best_model*.pth', recursive=True))
restore_path = ckpts[-1] if ckpts else None
if restore_path:
    print(f'체크포인트에서 재개: {restore_path}')

trainer = Trainer(
    TrainerArgs(restore_path=restore_path, skip_train_epoch=False),
    config,
    output_path=OUTPUT_DIR,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()

## Step 7: 음성 합성 테스트

In [ ]:
import glob, os
from TTS.api import TTS
from IPython.display import Audio, display

# 최신 체크포인트 찾기
ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/**/best_model*.pth', recursive=True))
configs = sorted(glob.glob(f'{OUTPUT_DIR}/**/config.json', recursive=True))

if not ckpts:
    print('체크포인트 없음 — Step 6 훈련 먼저 실행')
else:
    model_path = ckpts[-1]
    config_path = configs[-1]
    print(f'모델: {model_path}')

    tts = TTS(model_path=model_path, config_path=config_path, progress_bar=False)

    test_texts = [
        '안녕하세요 반갑습니다.',
        '오늘 날씨가 정말 좋네요.',
        '한국어 음성 합성 테스트입니다.',
    ]
    for text in test_texts:
        out_path = f'/tmp/test_{hash(text) % 10000}.wav'
        tts.tts_to_file(text=text, file_path=out_path)
        print(f'\n"{text}"')
        display(Audio(out_path, autoplay=False))

## Step 8: ONNX 내보내기 (sherpa-onnx 연결용)

훈련 완료 후 아래 셀로 ONNX 변환 → sherpa-onnx 서버에 바로 사용 가능

In [ ]:
import glob, os, torch
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer

ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/**/best_model*.pth', recursive=True))
if not ckpts:
    print('체크포인트 없음'); raise SystemExit

model_path = ckpts[-1]
config_path = sorted(glob.glob(f'{OUTPUT_DIR}/**/config.json', recursive=True))[-1]

config = VitsConfig()
config.load_json(config_path)
ap = AudioProcessor.init_from_config(config)
tokenizer, config = TTSTokenizer.init_from_config(config)

model = Vits(config, ap, tokenizer)
model.load_checkpoint(config, model_path, eval=True)
model.cuda()

# ONNX 내보내기
onnx_path = f'{OUTPUT_DIR}/my-korean-voice.onnx'
model.export_onnx(onnx_path)

print(f'✓ ONNX 저장: {onnx_path}  ({os.path.getsize(onnx_path)/1e6:.1f} MB)')
print('→ 이 파일을 sherpa-onnx 서버의 MODEL_DIR에 넣으면 사용 가능합니다')

---
## 훈련 가이드

| 데이터 | 에폭 | T4 예상 시간 |
|--------|------|-------------|
| 샘플 10개 (테스트) | 100  | ~5분 |
| 1시간 (1,000문장) | 1,000 | ~3시간 |
| 1시간 (1,000문장) | 5,000 | ~15시간 |

**팁:** Colab 세션은 12시간 제한 → Drive 체크포인트에서 자동 재개됩니다.  
3,000 에폭 이후부터 음질이 쓸 만해집니다.

## 내 목소리 데이터 녹음 가이드

- 조용한 환경, 일정한 거리 유지
- 각 문장 3~7초 내외
- 22050Hz, mono, 16-bit WAV로 저장
- 최소 500문장, 권장 1,000~3,000문장
- macOS: `rec -r 22050 -c 1 -b 16 output.wav`
- 또는 Audacity 사용 (무료)